# 02. Treinar Splink (dedupe cross-source)

Profile, blocking pré-treino, treino do modelo, predict e clustering.

A coorte não entra aqui: o modelo é treinado sem ver os rótulos, e a avaliação
roda em [`03_validar_coorte.ipynb`](03_validar_coorte.ipynb) sobre o modelo salvo.

**EDA descritiva:** [`01_analise_descritiva.ipynb`](01_analise_descritiva.ipynb).

In [9]:
import sys
from pathlib import Path

PROB_DIR = Path.cwd()
if PROB_DIR.name == 'notebooks':
    PROB_DIR = PROB_DIR.parent
if str(PROB_DIR) not in sys.path:
    sys.path.insert(0, str(PROB_DIR))

import pandas as pd
from config import (
    DUCKDB_MEMORY_LIMIT,
    DUCKDB_THREADS,
    OUTPUT_DIR,
    SPLINK_CLUSTERS,
    SPLINK_INPUT_VIEW,
    SPLINK_MODEL_JSON,
    SPLINK_PREDICTIONS,
    TABELA_LIMPA,
    USE_PHONETIC_STRIP_VOWELS,
    get_connection,
    get_splink_db_api,
    materialize_splink_input,
    print_paths,
    require_tables,
)

print_paths()
con = get_connection()
require_tables(con, [TABELA_LIMPA], notebook_origem='00b')
materialize_splink_input(con)
db_api = get_splink_db_api(con)

n_reg = con.execute(f'SELECT COUNT(*) FROM {SPLINK_INPUT_VIEW}').fetchone()[0]
duck_settings = con.execute(
    "SELECT current_setting('threads'), current_setting('memory_limit')"
).fetchone()
print(f'Registros: {n_reg:,}')
print(
    f'DuckDB: threads={duck_settings[0]}, memory_limit={duck_settings[1]} '
    f'(defaults: {DUCKDB_THREADS}, {DUCKDB_MEMORY_LIMIT})'
)

SPLINK_ANALYSIS_SAMPLE_N = 1_000_000
analysis_table = SPLINK_INPUT_VIEW
if n_reg > SPLINK_ANALYSIS_SAMPLE_N:
    con.execute(f'''
    CREATE OR REPLACE TEMP TABLE splink_analysis_sample AS
    SELECT * FROM {SPLINK_INPUT_VIEW}
    USING SAMPLE {SPLINK_ANALYSIS_SAMPLE_N} ROWS
    ''')
    analysis_table = 'splink_analysis_sample'
    print(f'Amostra profile/blocking: {SPLINK_ANALYSIS_SAMPLE_N:,} de {n_reg:,}')


OUTPUT_DIR: /home/ibge.gov.br/ramon.goncalves/data/probabilistico_output
CPF_ARQUIVO: /home/ibge.gov.br/ramon.goncalves/singed/bases/bronze/cpf/cpf.parquet
CENSO_PESSOAS_ARQUIVO: /home/ibge.gov.br/ramon.goncalves/singed/bases/bronze/censo/censo_pessoas_2022_20260505.parquet
CENSO_CEP_ARQUIVO: /home/ibge.gov.br/ramon.goncalves/singed/bases/raw/censo/data_cep_uniq.csv
COHORT_DEDUP_ARQUIVO: /home/ibge.gov.br/ramon.goncalves/capefe/dados/CohortDados/cohort_dedup.parquet
FILTRO_UF: None
FILTRO_MUNICIPIO: 2111300
USE_PHONETIC_STRIP_VOWELS: False
ANO_OBITO_CORTE: 2021
ANO_NASCIMENTO_MIN: 1900
SEXO_VALIDOS: ('M', 'F')
DUCKDB_ARQUIVO: /home/ibge.gov.br/ramon.goncalves/data/probabilistico_output/probabilistico.duckdb
splink_input → registro_limpo
Registros: 2,476,718
DuckDB: threads=20, memory_limit=279.3 GiB (defaults: 20, 300GB)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Amostra profile/blocking: 1,000,000 de 2,476,718


## Sanity check - composição da base

Volume por fonte e preenchimento das colunas usadas no linkage. Blocking em
coluna muito vazia gera poucos pares candidatos.

In [10]:
from IPython.display import display

display(con.execute(f'''
SELECT origem, COUNT(*) AS n
FROM {SPLINK_INPUT_VIEW} GROUP BY 1 ORDER BY 2 DESC
''').df())

LINKAGE_COLS = [
    'primeiro_nome', 'ultimo_nome', 'nome_completo','nome_meio',
    'nome_mae', 'primeiro_nome_mae', 'nome_meio_mae', 'ultimo_nome_mae',
    'data_nascimento', 'idade', 'cep', 'sexo', 'uf',
    'nome_completo_phon','primeiro_nome_phon','nome_meio_phon','ultimo_nome_phon',
    'nome_mae_phon','primeiro_nome_mae_phon','nome_meio_mae_phon','ultimo_nome_mae_phon'
]

cols_presentes = [
    c for c in LINKAGE_COLS
    if c in set(con.execute(f'SELECT * FROM {SPLINK_INPUT_VIEW} LIMIT 0').df().columns)
]
preenchimento = ',\n    '.join(
    f"ROUND(100.0 * COUNT({c}) / COUNT(*), 1) AS pct_{c}" for c in cols_presentes
)
display(con.execute(f'''
SELECT origem, {preenchimento}
FROM {SPLINK_INPUT_VIEW} GROUP BY origem ORDER BY origem
''').df().T)


,origem,n
0,cpf,1438943
1,censo,1037775


,0,1
origem,censo,cpf
pct_primeiro_nome,95.6,100.0
pct_ultimo_nome,95.1,100.0
pct_nome_completo,95.6,100.0
pct_nome_meio,79.7,97.2
pct_nome_mae,28.9,95.8
pct_primeiro_nome_mae,28.9,95.8
pct_nome_meio_mae,24.2,90.6
pct_ultimo_nome_mae,28.9,95.8
pct_data_nascimento,83.3,97.7


## Exploração pré-modelo

Profile Splink das colunas de linkage e análise de blocking (cumulativo + maiores blocos).

In [11]:
from splink import block_on
from splink.blocking_analysis import cumulative_comparisons_to_be_scored_from_blocking_rules_chart
from splink.exploratory import profile_columns
from splink.blocking_analysis import n_largest_blocks
# Blocking rules — colunas completas (sem substr)
blocking_rules = [
    block_on('primeiro_nome_phon', 'ultimo_nome_phon'),
    block_on('ultimo_nome_phon', 'data_nascimento'),
    block_on('primeiro_nome_phon', 'data_nascimento'),
    block_on('cep', 'ultimo_nome_phon'),
    block_on('cep', 'primeiro_nome_phon'),    
]

profile_columns(
    con.execute(f'SELECT * FROM {analysis_table}').df(),
    db_api,
    column_expressions=[
        'primeiro_nome_phon', 'ultimo_nome_phon', 'nome_completo_phon',
        'cep', 'data_nascimento', 'idade',
    ],
)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

alt.VConcatChart(...)

In [12]:
cumulative_comparisons_to_be_scored_from_blocking_rules_chart(
    table_or_tables=analysis_table,
    blocking_rules=blocking_rules,
    db_api=db_api,
    link_type='dedupe_only',
)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

alt.Chart(...)

## Modelo Splink

Settings e `Linker` sobre a base completa (`SPLINK_INPUT_VIEW`).

In [8]:
from splink import Linker, SettingsCreator
import splink.comparison_library as cl

input_cols = set(con.execute(f'SELECT * FROM {SPLINK_INPUT_VIEW} LIMIT 0').df().columns)

comparisons = [
    cl.NameComparison('nome_completo_phon').configure(term_frequency_adjustments=True),
    cl.ExactMatch('primeiro_nome_phon').configure(term_frequency_adjustments=True),
    cl.ExactMatch('nome_meio_phon').configure(term_frequency_adjustments=True),
    cl.ExactMatch('ultimo_nome_phon').configure(term_frequency_adjustments=True),    
    cl.NameComparison('nome_mae_phon').configure(term_frequency_adjustments=True),
    cl.ExactMatch('primeiro_nome_mae_phon').configure(term_frequency_adjustments=True),
    cl.ExactMatch('nome_meio_mae_phon').configure(term_frequency_adjustments=True),
    cl.ExactMatch('ultimo_nome_mae_phon').configure(term_frequency_adjustments=True),    
    cl.DateOfBirthComparison('data_nascimento', input_is_string=True),
    cl.ExactMatch('idade'),
    cl.ExactMatch('sexo').configure(term_frequency_adjustments=True),
    #cl.ExactMatch('uf').configure(term_frequency_adjustments=True),    
]
if USE_PHONETIC_STRIP_VOWELS and 'nome_completo_phon_sv' in input_cols:
    comparisons.append(cl.NameComparison('nome_completo_phon_sv'))

settings = SettingsCreator(
    link_type='dedupe_only',
    unique_id_column_name='unique_id',
    comparisons=comparisons,
    blocking_rules_to_generate_predictions=blocking_rules,
    retain_intermediate_calculation_columns=True,
)
linker = Linker(SPLINK_INPUT_VIEW, settings, db_api=db_api)

SETTINGS VALIDATION: Errors were identified in your settings dictionary. 

Invalid Columns(s) in Comparison(s)

Comparison: nome_meio_phon
--------------------------------------
    SQL: `"nome_meio_phon_l" IS NULL OR "nome_meio_phon_r" IS NULL`
       - Missing column(s) from input dataframe(s): `nome_meio_phon`

    SQL: `"nome_meio_phon_l" = "nome_meio_phon_r"`
       - Missing column(s) from input dataframe(s): `nome_meio_phon`

Comparison: nome_meio_mae_phon
--------------------------------------
    SQL: `"nome_meio_mae_phon_l" IS NULL OR "nome_meio_mae_phon_r" IS NULL`
       - Missing column(s) from input dataframe(s): `nome_meio_mae_phon`

    SQL: `"nome_meio_mae_phon_l" = "nome_meio_mae_phon_r"`
       - Missing column(s) from input dataframe(s): `nome_meio_mae_phon`

You may want to verify your settings dictionary has valid inputs in all fields before continuing.


In [5]:
deterministic_rules = [
    block_on('primeiro_nome', 'ultimo_nome', 'data_nascimento'),
    block_on('nome_completo', 'data_nascimento'),
]
linker.training.estimate_probability_two_random_records_match(deterministic_rules, recall=0.7)
linker.training.estimate_u_using_random_sampling(max_pairs=2_000_000)
linker.training.estimate_parameters_using_expectation_maximisation(block_on('data_nascimento'),estimate_without_term_frequencies=True)


Probability two random records match is estimated to be  2.14e-07.
This means that amongst all possible pairwise record comparisons, one in 4,671,944.45 are expected to match.  With 3,067,064,787,403 total possible comparisons, we expect a total of around 656,485.71 matching pairs
----- Estimating u probabilities using random sampling -----

Estimated u probabilities using random sampling

Your model is not yet fully trained. Missing estimates for:
    - nome_completo (no m values are trained).
    - primeiro_nome (no m values are trained).
    - nome_meio (no m values are trained).
    - ultimo_nome (no m values are trained).
    - nome_completo_phon (no m values are trained).
    - data_nascimento (no m values are trained).
    - idade (no m values are trained).
    - nome_mae (no m values are trained).
    - primeiro_nome_mae (no m values are trained).
    - nome_meio_mae (no m values are trained).
    - ultimo_nome_mae (no m values are trained).
    - sexo (no m values are trained)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


----- Starting EM training session -----

Estimating the m probabilities of the model by blocking on:
l."data_nascimento" = r."data_nascimento"

Parameter estimates will be made for the following comparison(s):
    - nome_completo
    - primeiro_nome
    - nome_meio
    - ultimo_nome
    - nome_completo_phon
    - idade
    - nome_mae
    - primeiro_nome_mae
    - nome_meio_mae
    - ultimo_nome_mae
    - sexo
    - cep

Parameter estimates cannot be made for the following comparison(s) since they are used in the blocking rules: 
    - data_nascimento


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


Level All other comparisons on comparison idade not observed in dataset, unable to train m value

Iteration 1: Largest change in params was -0.647 in the m_probability of cep, level `Exact match on cep`
Iteration 2: Largest change in params was 0.118 in the m_probability of nome_meio_mae, level `All other comparisons`
Iteration 3: Largest change in params was 0.179 in the m_probability of nome_completo_phon, level `Jaro-Winkler distance of nome_completo_phon >= 0.7`
Iteration 4: Largest change in params was 0.26 in the m_probability of nome_meio_mae, level `All other comparisons`
Iteration 5: Largest change in params was -0.198 in the m_probability of primeiro_nome, level `Exact match on primeiro_nome`
Iteration 6: Largest change in params was 0.119 in the m_probability of primeiro_nome, level `All other comparisons`
Iteration 7: Largest change in params was 0.141 in the m_probability of nome_completo, level `All other comparisons`
Iteration 8: Largest change in params was 0.262 in th

<EMTrainingSession, blocking on l."data_nascimento" = r."data_nascimento", deactivating comparisons data_nascimento>

In [6]:
linker.training.estimate_parameters_using_expectation_maximisation(block_on('primeiro_nome', 'ultimo_nome','cep'),estimate_without_term_frequencies=True)


----- Starting EM training session -----

Estimating the m probabilities of the model by blocking on:
(l."primeiro_nome" = r."primeiro_nome") AND (l."ultimo_nome" = r."ultimo_nome") AND (l."cep" = r."cep")

Parameter estimates will be made for the following comparison(s):
    - nome_completo
    - nome_meio
    - nome_completo_phon
    - data_nascimento
    - idade
    - nome_mae
    - primeiro_nome_mae
    - nome_meio_mae
    - ultimo_nome_mae
    - sexo

Parameter estimates cannot be made for the following comparison(s) since they are used in the blocking rules: 
    - primeiro_nome
    - ultimo_nome
    - cep


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


Iteration 1: Largest change in params was -0.958 in the m_probability of nome_completo, level `All other comparisons`
Iteration 2: Largest change in params was -0.585 in the m_probability of data_nascimento, level `Exact match on date of birth`
Iteration 3: Largest change in params was 0.447 in the m_probability of nome_meio, level `All other comparisons`
Iteration 4: Largest change in params was 0.592 in the m_probability of nome_completo_phon, level `Jaro-Winkler distance of nome_completo_phon >= 0.7`
Iteration 5: Largest change in params was 0.12 in probability_two_random_records_match
Iteration 6: Largest change in params was 0.0188 in probability_two_random_records_match
Iteration 7: Largest change in params was 0.0239 in probability_two_random_records_match
Iteration 8: Largest change in params was 0.00917 in probability_two_random_records_match
Iteration 9: Largest change in params was 0.00333 in probability_two_random_records_match
Iteration 10: Largest change in params was 0.

<EMTrainingSession, blocking on (l."primeiro_nome" = r."primeiro_nome") AND (l."ultimo_nome" = r."ultimo_nome") AND (l."cep" = r."cep"), deactivating comparisons primeiro_nome, ultimo_nome, cep>

In [7]:
SPLINK_MODEL_JSON.parent.mkdir(parents=True, exist_ok=True)
linker.misc.save_model_to_json(str(SPLINK_MODEL_JSON), overwrite=True)
print('Modelo salvo:', SPLINK_MODEL_JSON)

Modelo salvo: /home/ibge.gov.br/ramon.goncalves/data/probabilistico_output/splink_model.json


## Visualização pós-treino

Match weights e registros difíceis de linkar (unlinkables).

In [15]:
linker.visualisations.match_weights_chart()


/opt/venvs/singed/jhub/lib64/python3.9/site-packages/altair/vegalite/v6/api.py:4124: UserWarning: Automatically deduplicated selection parameter with identical configuration. If you want independent parameters, explicitly name them differently (e.g., name='param1', name='param2'). See https://github.com/vega/altair/issues/3891
  return _tp.from_dict(dct, validate=validate)


alt.VConcatChart(...)

In [16]:
linker.evaluation.unlinkables_chart()

alt.LayerChart(...)

## Predict + clustering

In [37]:
df_predict = linker.inference.predict(threshold_match_probability=0.5)
df_predictions = df_predict.as_pandas_dataframe()
clusters = linker.clustering.cluster_pairwise_predictions_at_threshold(
    df_predict, threshold_match_probability=0.95,
)
df_clusters = clusters.as_pandas_dataframe()
print('Pares:', len(df_predictions), 'Clusters:', df_clusters['cluster_id'].nunique())



FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Blocking time: 122.68 seconds


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Predict time: 891.78 seconds

 -- WARNING --
You have called predict(), but there are some parameter estimates which have neither been estimated or specified in your settings dictionary.  To produce predictions the following untrained trained parameters will use default values.
Comparison: 'nome_completo':
    m values not fully trained
Comparison: 'nome_completo':
    u values not fully trained
Comparison: 'primeiro_nome':
    m values not fully trained
Comparison: 'primeiro_nome':
    u values not fully trained
Comparison: 'nome_meio':
    m values not fully trained
Comparison: 'nome_meio':
    u values not fully trained
Comparison: 'ultimo_nome':
    m values not fully trained
Comparison: 'ultimo_nome':
    u values not fully trained
Comparison: 'nome_completo_phon':
    m values not fully trained
Comparison: 'nome_completo_phon':
    u values not fully trained
Comparison: 'data_nascimento':
    m values not fully trained
Comparison: 'data_nascimento':
    u values not fully trained

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

...

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Completed iteration 2, num edges remaining to process: 420762
Completed iteration 3, num edges remaining to process: 288052
Completed iteration 4, num edges remaining to process: 135446
Completed iteration 5, num edges remaining to process: 108312
Completed iteration 6, num edges remaining to process: 34228
Completed iteration 7, num edges remaining to process: 9134
Completed iteration 8, num edges remaining to process: 2204
Completed iteration 9, num edges remaining to process: 358
Completed iteration 10, num edges remaining to process: 50
Completed iteration 11, num edges remaining to process: 0


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Pares: 3404900 Clusters: 1316221


## Waterfall — amostra de pares

Contribuição de cada comparação ao match weight (5 pares).

In [38]:
records_to_plot = df_predictions.head(5).to_dict(orient="records")
linker.visualisations.waterfall_chart(records_to_plot, filter_nulls=False)


#records_sample = df_predictions.to_dict(orient='records')
#Linker.visualisations.waterfall_chart(records_sample, filter_nulls=False)

alt.LayerChart(...)

In [39]:
#df_predictions.head(5)
#df_predictions['unique_id_l'].str.contains("censo").head(5)
df_predictions[(df_predictions['unique_id_l'].str.contains("censo")) & df_predictions['unique_id_r'].str.contains("cpf")].head(5)

,match_weight,match_probability,unique_id_l,unique_id_r,nome_completo_l,nome_completo_r,gamma_nome_completo,tf_nome_completo_l,tf_nome_completo_r,bf_nome_completo,...,gamma_sexo,tf_sexo_l,tf_sexo_r,bf_sexo,bf_tf_adj_sexo,cep_l,cep_r,gamma_cep,bf_cep,match_key
282,29.585608,1.000000,censo_2111300050000960010020000039000001000000001,cpf_28895053320,MARIA JOSE BARBOSA,MARIA JOSE BARBOSA,4,0.000012,1.234055e-05,1024.000000,...,1,0.519685,0.519685,1024.0,0.001785,65030430,65000000,0,0.03125,0
283,12.126164,0.999776,censo_2111300050000980010090000330000001000000001,cpf_10009299300,JOAO CARLOS DE SOUSA,JOAO CARLOS COSTA DE SOUSA,3,0.000004,8.227033e-07,8.000000,...,1,0.480315,0.480315,1024.0,0.001932,65031422,65030030,0,0.03125,0
285,0.674468,0.614793,censo_2111300050001320020010000115000001000000004,cpf_62237959390,JOAO LUCAS PEREIRA,JOAO LUCAS GALVAO PEREIRA,2,0.000002,1.234055e-06,1.259921,...,1,0.480315,0.480315,1024.0,0.001932,65082507,65000000,0,0.03125,0
286,25.255596,1.000000,censo_2111300050000960030010000233000001000000001,cpf_09551492315,MARIA DA CONCEICAO FERREIRA,MARIA DA CONCEICAO FERREIRA,4,0.000026,2.632651e-05,1024.000000,...,1,0.519685,0.519685,1024.0,0.001785,65031060,65026010,0,0.03125,0
287,27.545362,1.000000,censo_2111300050000950070010000204000003000000001,cpf_22493557387,JOAO BATISTA SANTOS,JOAO BATISTA SANTOS,4,0.000019,1.933353e-05,1024.000000,...,1,0.480315,0.480315,1024.0,0.001932,65031160,65040780,0,0.03125,0


## Cluster studio

Dashboard interativo para inspecionar clusters (amostra por tamanho).

In [40]:
from IPython.display import IFrame, display

CLUSTER_STUDIO_HTML = OUTPUT_DIR / 'dashboards' / 'cluster_studio.html'
CLUSTER_STUDIO_HTML.parent.mkdir(parents=True, exist_ok=True)

linker.visualisations.cluster_studio_dashboard(
    df_predict,
    clusters,
    str(CLUSTER_STUDIO_HTML),
    sampling_method='by_cluster_size',
    overwrite=True,
)
print('Dashboard:', CLUSTER_STUDIO_HTML)
display(IFrame(src=str(CLUSTER_STUDIO_HTML), width='100%', height=1200))

Dashboard: /home/ibge.gov.br/ramon.goncalves/data/probabilistico_output/dashboards/cluster_studio.html


## Diagnóstico dos scores (sem rótulos)

Distribuição de match weight e tamanho de cluster. Precision/recall contra a
coorte ficam no NB03, sobre o modelo salvo acima.

In [41]:
display(df_predictions['match_probability'].describe())
faixas = pd.cut(df_predictions['match_weight'], bins=20)
display(
    df_predictions.groupby(faixas, observed=True)
    .size()
    .rename('n_pares')
    .to_frame()
)

count    3.404900e+06
mean     9.528958e-01
std      1.153990e-01
min      5.000017e-01
25%      9.987734e-01
50%      1.000000e+00
75%      1.000000e+00
max      1.000000e+00
Name: match_probability, dtype: float64

,n_pares
match_weight,
"(-0.173, 8.672]",813884
"(8.672, 17.344]",366602
"(17.344, 26.015]",636802
"(26.015, 34.687]",686209
"(34.687, 43.359]",322542
"(43.359, 52.031]",115433
"(52.031, 60.702]",111974
"(60.702, 69.374]",117167
"(69.374, 78.046]",79128


In [42]:
tamanhos = df_clusters.groupby('cluster_id').size()
print(f'Clusters: {tamanhos.size:,} | maior: {tamanhos.max():,} | singletons: {(tamanhos == 1).sum():,}')
display(
    tamanhos.value_counts().sort_index().head(20)
    .rename('n_clusters').to_frame().rename_axis('tamanho')
)

# Clusters grandes demais indicam blocking/threshold frouxo — inspecionar antes do NB03.
display(tamanhos.sort_values(ascending=False).head(10).rename('tamanho').to_frame())

Clusters: 1,316,221 | maior: 270,029 | singletons: 795,819


,n_clusters
tamanho,
1,795819
2,378020
3,53548
4,48344
5,12779
6,12002
7,4440
8,3491
9,1931


,tamanho
cluster_id,
censo_2111300050000010010020000025000003000000001,270029
censo_2111300050000030150040000693000001000000002,140
censo_2111300050000430020040000230000001000000002,129
censo_2111300050000080050030000614000001000000001,118
cpf_63889132375,114
censo_2111300050000430010010000237000001000000001,101
censo_2111300050000420050010000275000001000000002,101
censo_2111300050000570020030000113000001000000003,99
censo_2111300050000290060140000321000001000000002,88


In [43]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
df_predictions.to_parquet(SPLINK_PREDICTIONS)
df_clusters.to_parquet(SPLINK_CLUSTERS)
print('Predictions:', SPLINK_PREDICTIONS)
print('Clusters:', SPLINK_CLUSTERS)

Predictions: /home/ibge.gov.br/ramon.goncalves/data/probabilistico_output/splink_predictions.parquet
Clusters: /home/ibge.gov.br/ramon.goncalves/data/probabilistico_output/splink_clusters.parquet


## Encerrar

Artefatos prontos para o [`03_validar_coorte.ipynb`](03_validar_coorte.ipynb).

In [44]:
for label, p in [
    ('modelo', SPLINK_MODEL_JSON),
    ('predictions', SPLINK_PREDICTIONS),
    ('clusters', SPLINK_CLUSTERS),
]:
    print(f'{label:12s} {"ok " if p.exists() else "FALTA"} {p}')

con.close()


modelo       ok  /home/ibge.gov.br/ramon.goncalves/data/probabilistico_output/splink_model.json
predictions  ok  /home/ibge.gov.br/ramon.goncalves/data/probabilistico_output/splink_predictions.parquet
clusters     ok  /home/ibge.gov.br/ramon.goncalves/data/probabilistico_output/splink_clusters.parquet
